# Create a ``config.ini``

## Generate default values

In most case, whether you're new to ForMoSA or not, we **strongly** recommend using this method to generate your first config file.
1) Download this empty config file : [config.ini 📥](config.ini).
2) Set up your paths (i.e., tell ForMoSA where to find the data and model files, and where to store the results).
3) Launch ``global_file.py`` with this config file. This will automatically create a reference config file (``config_ref.ini``) in ``result_path`` with all missing inputs; i.e. :

In [ ]:
from ForMoSA.global_file import GlobFile
config_path = '/home/mravet/Documents/These/FORMOSA/OUTPUTS/Channel1/config.ini'
GlobFile(config_path) # => Will create a complete config file with all default values

4) You can now modify ``config_ref.ini`` with the parameters you want ! <u>Be careful :</u> you must at least define the prior values for the grid parameters before running an inversion. Check next section to learn more about each **parameters**.

***
**Why bother with this ?** 

In future updates, some parameters may be added or removed. Generating a new config ensures that you'll always have access to all available parameters provided by ForMoSA  ☺️ !

***

## Parameters *(version 1.1.5)*

List and description of each config file parameter. If, for a given parameter, '<u>MOSAIC :</u> Yes', it means that the parameter can be defined separatly for each observation (i.e ``parameter`` = par1, par2, ...)

### ``[config_adapt]``

- ``method`` = Interpolation method for the grid.

    <u>format :</u> ``'linear'`` or ``'nearest'`` or ``'zero'`` or ``'slinear'`` or ``'quadratic'`` or ``'cubic'`` or ``'quintic'`` or ``'pchip'`` or ``'barycentric'`` or ``'krogh'`` or ``'akima'`` or ``'makima'``

    <u>comments :</u> ``'linear'`` will be used by default. ⚠️ For safety reasons, this will also be the interpolation method during the nested sampling. Refer to the [xarray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.interp.html) documentation for more details.

    <u>MOSAIC :</u> No


- ``emulator`` = Emulator to use.

    <u>format :</u> ``'NA'`` or ``'PCA', ncomp`` or ``'NMF', ncomp``

    <u>comments :</u> ``'NA'`` will be used by default, meaning the grid won't be emulated. ``ncomp`` is the number of components used for the PCA or NMF decomposition. ⚠️  You cannot set more components than the number of spectral channels. Refer to the [scikit-learn](https://scikit-learn.org/stable/api/sklearn.decomposition.html) documentation for more details.

    <u>MOSAIC :</u> No (will soon change)

- ``target_res_obs`` = Observation resolution to reach.

    <u>format :</u> ``'obs'`` or ``float``

    <u>comments :</u> ``'obs'`` will be used by default, meaning ForMoSA won't change the observation resolution. If a float is given, ForMoSA will take the minimum between this value, the resolution of the model and the resolution of the observation.

    <u>MOSAIC :</u> Yes

- ``target_res_mod`` = Model resolution to reach (prior to the inversion).

    <u>format :</u> ``'obs'`` or ``'mod'`` or ``float``

    <u>comments :</u> ``'obs'`` will be used by default, meaning ForMoSA will convolve and resample the model to the observation resolution prior to the inversion. If ``'mod'``, ForMoSA won't change the model resolution prior to the inversion (usefull for high resolution spectroscopy). If a float is given, ForMoSA will take the minimum between this value and the resolution of the model.

    <u>MOSAIC :</u> Yes

- ``res_cont`` = Approximate resolution of the continuum.

    <u>format :</u> ``'NA'`` or ``float``

    <u>comments :</u> ``'NA'`` will be used by default, meaning ForMoSA won't remove the continuum. If a float is given, ForMoSA will remove a continuum extracted at this resolution. ⚠️ To use this parameter, you must also define ``wav_cont`` (see below).

    <u>MOSAIC :</u> Yes

- ``wav_cont`` = Wavelength range(s) used to estimate of the continuum.

    <u>format :</u> ``'NA'`` or ``'window1_min / window1_max, window2_min / ... / windowN_max'``

    <u>comments :</u> ``'NA'`` will be used by default, meaning ForMoSA won't remove the continuum. If a wavelength range (in µm) is provided, ForMoSA will remove a continuum estimated using this range. ⚠️ To use this parameter, you must also define ``res_cont`` (see above). Separate windows can be defined using ``'/'`` as separator.

    <u>MOSAIC :</u> Yes

### ``[config_inversion]``

- ``logL_type`` = Log-likelihood used during inversion.

    <u>format :</u> ``'chi2'`` or ``'chi2_covariance'`` or ``'chi2_noisescaling'`` or ``'chi2_noisescaling_covariance'`` or ``'CCF_Brogi'`` or ``'CCF_Zucker'`` or ``'CCF_custom'``

    <u>comments :</u> ``'chi2'`` will be used by default. This defines the core of the inversion. The choice of likelihood function entirely depends on the type of data you're dealing with. ⚠️ Photometric points are always compared to the model using ``'chi2'``, regardless of the likelihood selected for spectroscopy.

    <u>MOSAIC :</u> Yes

    <u>exemple: </u>

***
``'chi2'`` : $~~~log(L)=-\frac{\chi^2}{2}=-\frac{1}{2}\sum\left(\frac{d-m}{\sigma}\right)^2$

``'chi2_covariance'`` : $~~~log(L)=-\frac{\chi^2}{2}=-\frac{1}{2}(d-m)^T C^{-1} (d-m)$

``'chi2_noisescaling'`` : $~~~log(L)=-\frac{N}{2}log\left(\frac{\chi^2}{N}\right)=-\frac{N}{2}log\left(\sum\frac{1}{N}\left(\frac{d-m}{\sigma}\right)^2\right)$ [Ruffio et al. (2019)](https://ui.adsabs.harvard.edu/abs/2019AJ....158..200R/abstract)

``'chi2_noisescaling_covariance'`` : $~~~log(L)=-\frac{N}{2}log\left(\frac{\chi^2}{N}\right)=-\frac{N}{2}log\left(\frac{1}{N}(d-m)^T C^{-1} (d-m)\right)$ [Ruffio et al. (2019)](https://ui.adsabs.harvard.edu/abs/2019AJ....158..200R/abstract)

``'CCF_Brogi'`` : $~~~log(L)=-\frac{1}{2}log\left(S_f^2 + S_m^2 - 2R\right)=-\frac{1}{2}log\left(\frac{1}{N}\sum d^2 + \frac{1}{N}\sum m^2 - 2\frac{1}{N}\sum dm \right)$ [Brogi et al. (2019)](https://ui.adsabs.harvard.edu/abs/2019AJ....157..114B/abstract)

``'CCF_Zucker'`` : $~~~log(L)=-\frac{1}{2}log\left(1 - C^2\right)=-\frac{1}{2}log\left(1 - \frac{\sum dm}{\sum d^2 \sum m²}\right)$ [Zucker et al. (2003)](https://ui.adsabs.harvard.edu/abs/2003MNRAS.342.1291Z/abstract)

``'CCF_custom'`` : $~~~log(L)=-\frac{1}{2\bar\sigma}\left(S_f^2 + S_m^2 - 2R\right)=-\frac{1}{2} \left(\frac{1}{N}\sum\frac{1}{\sigma^2} \right)  \left(\frac{1}{N}\sum d^2 + \frac{1}{N}\sum m^2 - 2\frac{1}{N}\sum dm \right)$

... with $d$ and $m$ being the data and model arrays, respectively. $\sigma$ represents the error bars, and $C$ is the covariance matrix.

***

- ``wav_fit`` = Wavelength range(s) used during the nested sampling procedure.

    <u>format :</u> ``'window1_min / window1_max, window2_min / ... / windowN_max'``

    <u>comments :</u> ``'0,100'`` will be used by default. If a wavelength range (in µm) is provided, ForMoSA will ignore all points outside of it during the inversion. Separate windows can be defined using ``'/'`` as separator.

    <u>MOSAIC :</u> Yes

- ``ns_algo`` = Nested sampling algorithm.

    <u>format :</u> ``'nestle'`` or ``'pymultinest'`` or ``'ultranest'``

    <u>comments :</u> ``'nestle'`` will be used by default. We recommend using ``'pymultinest'`` for large datasets as it enables for efficient parallelisation. Refer to [nestle](http://kylebarbary.com/nestle/), [pymultinest](https://johannesbuchner.github.io/PyMultiNest/) and [ultranest](https://johannesbuchner.github.io/UltraNest/readme.html) documentations for more details.

    <u>MOSAIC :</u> No

- ``npoint`` = Number of living points during the nested sampling procedure.

    <u>format :</u> ``int``

    <u>comments :</u> ``100`` will be used by default.

    <u>MOSAIC :</u> No

### ``[config_highcont_models]``

- ``hc_type`` = Method to compute the high-contrast model.

    <u>format :</u> ``'NA'`` or ``'nofit_rm_spec'`` or ``'nonlinear_fit_spec'`` or ``'fit_spec'`` or ``'rm_spec'`` or ``'fit_spec_rm_cont'`` or ``'fit_spec_fit_cont'``

    <u>comments :</u> ``'NA'`` will be used by default, meaning ForMoSA won't fit for stellar contamination. This defines the function used to compute the high-contrast model. ⚠️ To use this parameter, you must also define ``hc_bound_lsq`` (see below). We strongly recommend that new users read the API documentation on high-contrast models before using these functions.

    <u>MOSAIC :</u> Yes

- ``hc_bounds_lsq`` = Least-square bounds.

    <u>format :</u> ``'NA'`` or ``'lower, upper'`` 
    
    <u>comments :</u> ``'NA'`` will be used by default, meaning ForMoSA won't fit for stellar contamination. This defines the bounds of the lsq used to compute the high-contrast model. ⚠️ To use this parameter, you must also define ``hc_bound_lsq`` (see below). We strongly recommend that new users read the API documentation on high-contrast models before using these functions.

    <u>MOSAIC :</u> Yes

### ``[config_parameters]``

***
This is where physics happens ! 😊. You'll be able to define priors and ranges for your physical parameters.

As of now, four functions can be used :
1) ``'uniform', lower, upper``
2) ``'loguniform', lower, uppper`` (⚠️ ``lower`` and ``upper`` must be >0)
3) ``'gaussian', mu, sigma``
4) ``'constant', value``

⚠️ For parameters available with MOSAIC, you should **ALWAYS** use three entries per observation (e.g. prior_1, value1_1, value2_1, prior_2, value1_2, value2_2, ..., prior_N, value1_N, value2_N)

![priors_teff_plot](../_static/priors_teff_plot.png)

***

- ``parN`` = N'th (from 1 up to 5) parameter of the atmospheric grid.

    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default. These are the main parameters of your inversion. They are specific to each atmospheric grid, but usually include quantities like $T_{eff}$, $log(g)$, $[M/H]$, ... ⚠️ Be sure to check the documentation of the grid to learn the number of parameters and their valid ranges.

    <u>MOSAIC :</u> No

- ``r`` = Radius in $R_{jup}$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default, meaning ForMoSA will do an analytical rescaling. You need to define both ``r`` and ``d`` so that ForMoSA can compute an analytical scaling factor ``ck=(r/d)²`` to rescale the model. ⚠️ This parameter cannot be used alongside high-contrast models, as they also apply a rescaling to the model.

    <u>MOSAIC :</u> No

- ``d`` = Distance in $pc$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default,meaning will do an analytical rescaling. You need to define both ``r`` and ``d`` so that ForMoSA can compute an analytical scaling factor ``ck=(r/d)²`` to rescale the model. ⚠️ This parameter cannot be used alongside high-contrast models, as they also apply a rescaling to the model.

    <u>MOSAIC :</u> No

- ``alpha`` = Scaling parameter.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default, meaning ``alpha=1``. You need to define both ``r`` and ``d`` so that ForMoSA can compute an analytical scaling factor ``ck=alpha*(r/d)²`` to rescale the model. ⚠️ This parameter cannot be used alongside high-contrast models, as they also apply a rescaling to the model. We also strongly recommend that you put constraining priors on this parameter as it is highly degenerated with both ``r`` and ``d``. If you have multiple observations and fit multiple ``alpha``s, setting one of them to ``'NA'`` will use the analytical computation of the scaling factor (usefull for continuum-substracted data).

    <u>MOSAIC :</u> Yes

    <u>exemple :</u>

***
![ck](../_static/ck.png)

***

- ``rv`` = Radial velocity in $km/s$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default. This parameter shifts the model in wavelength. To avoid edge effects during reinterpolation, if ``target_res_mod = 'mod'`` or ``target_res_mod = float``, the model is saved with a larger spectral extent during the adaptation step.

    <u>MOSAIC :</u> Yes

    <u>exemple :</u>

***
![rv](../_static/rv.png)

***

- ``av`` = Extinction coeficient in $mag$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default. This will apply interstellar extinction to the model. It can also be used to emulate missing cloud opacities in your grid.

    <u>MOSAIC :</u> No

    <u>exemple :</u>

***
![av](../_static/av.png)

***

- ``vsini`` = Projected rotational velocity in $km/s$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2, 'method'``

    <u>comments :</u> ``'NA'`` will be used by default. You need to define both ``vsini`` and ``ld`` so that ForMoSA can compute the broadening of the spectral lines. Constraints obtained on this parameter for observations at a resolution $<$100,000 are not robust for slow rotators. To avoid edge effects during reinterpolation, we also recommend to fit ``rv`` as well. Since this parameter can be computationally expensive to fit, ForMoSA allows you to choose between four methods : ``'RotBroad'`` or ``'FastRotBroad'`` or ``'Accurate'`` or ``'AccurateFastRotBroad'``. You should always specify your method **after** the priors. Please refer to the API documentation for more information.

    <u>MOSAIC :</u> Yes

- ``ld`` = Limb-darkening.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default. You need to define both ``vsini`` and ``ld`` so that ForMoSA can compute the broadening of the spectral lines. This parameter is usually hard to constrain so we recommend using a constant value.

    <u>MOSAIC :</u> Yes

    <u>exemple :</u>

***
![vsini](../_static/vsini.png)

***

- ``bb_t`` = Black-body temperature in $K$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default. You need to define ``bb_t``, ``bb_r`` and ``d`` so that ForMoSA can add the contribution from a simple black-body circumplanetary disk.

    <u>MOSAIC :</u> No

- ``bb_r`` = Black-body radius in $R_{jup}$.
    
    <u>format :</u> ``'NA'`` or  ``'prior', value1, value2``

    <u>comments :</u> ``'NA'`` will be used by default. You need to define ``bb_t``, ``bb_r`` and ``d`` so that ForMoSA can add the contribution from a simple black-body circumplanetary disk.

    <u>MOSAIC :</u> No

***
![bb](../_static/bb.png)

***

### ``[config_nestle]``

See [Nestle](http://kylebarbary.com/nestle/index.html) documentation

### ``[config_pymultinest]``

See [Pymultinest](https://github.com/JohannesBuchner/PyMultiNest/blob/master/pymultinest/run.py) documentation

### ``[config_ultranest]``

See [Ultranest](https://johannesbuchner.github.io/UltraNest/readme.htmll) documentation